[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/07_aggregation_trends.ipynb)

# Sitzung 7 — Aggregation & Trends

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI · 18.11.2026**

Bisher haben wir **einzelne** Bewertungen eingeordnet: Stimmung, Themen, verlässliches JSON. Eine Produktmanagerin liest aber keine 279 Einzelurteile. Sie will wissen: *Wird es besser oder schlechter? Und warum?* Heute machen wir aus Einzelurteilen **Aussagen**, und prüfen, ob diese Aussagen halten.

**Ablauf**
1. **Die Fragen einer Produktmanagerin (Abschnitt 1):** was wir heute beantworten wollen.
2. **Wird es schlechter? (Abschnitt 2):** Anzahl oder Anteil, und wie sicher ist ein Monatswert?
3. **Und die Sterne? (Abschnitt 3):** das billige Signal, und was es nicht kann.
4. **War es das Update? (Abschnitt 4):** Korrelation ist nicht Ursache, und wie man eine Ursache prüft.
5. **Eure Manager-Frage (Abschnitt 5):** selbst eine Frage beantworten, und sagen, was die Zahl *nicht* beweist.
6. **Der Bericht an Lena (Abschnitt 6):** alle Antworten auf einer Seite, der Vorläufer des Dashboards.

Heute brauchen wir **keinen API-Schlüssel**: Es wird nur gerechnet, nicht klassifiziert.

### Rückblick auf Sitzung 6

```text
1. Modelle liefern kein sicheres Format   -> erst herausschneiden, dann prüfen
2. Gültiges JSON != richtige Antwort      -> Schema als Vertrag (pydantic)
3. Schema prüft die Form, nicht den Sinn  -> Begründung lesen
4. Antwort ungültig                       -> Fehlermeldung zurück, höchstens 2 Korrekturen
5. Unbekannte Themen                      -> Clustering macht Vorschläge, ein Mensch entscheidet
```

Am Ende von Sitzung 6 hatte jede Bewertung feste Felder: Stimmung, Themen mit Polarität, Begründung. Heute **fassen wir diese Felder zusammen**.

## 0. Setup und Daten

Heute geht es ums **Zusammenfassen**, nicht ums Klassifizieren. Damit alle dieselben Zahlen sehen und kein Schlüssel nötig ist, nehmen wir Stimmung und Themen direkt aus der Wahrheit unseres Generators, so als hätte die Pipeline aus Sitzung 5 und 6 fehlerfrei gearbeitet. In Sitzung 8 setzen wir die echte Pipeline davor.

**Was passiert in der nächsten Zelle?** Wir laden den bereinigten Datensatz aus Sitzung 4, hängen Stimmung und Themen an und legen für jede Bewertung den Monat fest.

In [ ]:
import urllib.request, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
pd.set_option("display.max_colwidth", 90)

URL = "https://raw.githubusercontent.com/kevisback/bda-course/session-07/wise2627/notebooks/radar.py"
try:
    urllib.request.urlretrieve(URL, "radar.py")
except Exception as e:
    print("Download fehlgeschlagen, nutze lokale radar.py:", e)
import radar
importlib.reload(radar)
radar.setup(backend="mock")                     # heute kein LLM nötig

df, wahrheit = radar.lade_sauber()
df["sentiment"] = wahrheit.loc[df["review_id"], "true_sentiment"].values
df["themen"] = [radar.wahre_themen(wahrheit.loc[r, "text_wahr"]) for r in df["review_id"]]

ohne_datum = df["datum"].isna().sum()
df = df.dropna(subset=["datum"]).reset_index(drop=True)   # ohne Datum kein Zeitverlauf
df["monat"] = df["datum"].dt.to_period("M")

print(f"{len(df)} Bewertungen mit Datum ({ohne_datum} ohne Datum weggelassen), "
      f"vom {df['datum'].min():%d.%m.%Y} bis {df['datum'].max():%d.%m.%Y}")
df[["datum", "monat", "quelle", "sterne", "sentiment", "themen", "text"]].head()

**Wie lese ich das?** Eine Zeile ist eine Bewertung. `themen` ist eine Liste aus Paaren (Thema, Polarität), wie in Sitzung 5. 13 Bewertungen haben kein Datum (Sitzung 4); für einen Zeitverlauf können wir sie nicht verwenden. Das ist ein kleiner Verlust, aber er gehört in jede Auswertung: *„266 von 279 Bewertungen"*, nicht *„alle"*.

**Was passiert in der nächsten Zelle?** Die einfachste Zusammenfassung überhaupt: Wie verteilen sich alle 266 Bewertungen auf die vier Stimmungen? Das ist die erste Zahl, die eine Managerin sehen will.

In [ ]:
REIHENFOLGE = ["positive", "mixed", "neutral", "negative"]
FARBEN = {"positive": "#16a34a", "mixed": "#f59e0b", "neutral": "#9ca3af", "negative": "#dc2626"}

verteilung = df["sentiment"].value_counts().reindex(REIHENFOLGE)
ax = (verteilung / len(df)).plot(kind="barh", color=[FARBEN[s] for s in REIHENFOLGE], figsize=(7, 2.6))
for i, (k, n) in enumerate(zip(verteilung, [len(df)] * 4)):
    ax.annotate(f"{k / n:.0%}  ({k})", (k / n + 0.01, i), va="center", fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_xlim(0, 0.6)
ax.set_title("Alle Bewertungen nach Stimmung"); ax.invert_yaxis(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

k = verteilung["negative"]
von, bis = radar.wilson(k, len(df))
print(f"Negativ insgesamt: {k} von {len(df)} = {k / len(df):.0%}  (Intervall {von:.0%} bis {bis:.0%})")

**Wie lese ich das?** 38 % aller Bewertungen sind negativ, mit Intervall etwa 33 % bis 44 % (Sitzung 3). Das ist ein guter erster Überblick, und gleichzeitig verschweigt die Zahl das Wichtigste: ob es **besser oder schlechter** wird. 38 % können ein stabiler Wert sein oder der Durchschnitt aus 10 % im Frühjahr und 80 % im Sommer. Um das zu sehen, müssen wir die Bewertungen nach der Zeit **gruppieren**.

## 1. Die Fragen einer Produktmanagerin

Lena ist Produktmanagerin für die Nimbus Q2 (unser fiktives Produkt). Sie hat drei Fragen:

1. **Wird es besser oder schlechter?**
2. **Sind die Sterne nicht genug?** Die gibt es ja schon, ohne Textanalyse.
3. **„Am 1. Juli kam App-Update 2.0. Seitdem geht es bergab. War es das Update?"**

Das Update ist ein Szenario für heute; im Datensatz steht davon nichts. Der Weg ist für alle drei Fragen derselbe:

```text
Einzelurteile  ->  gruppieren  ->  zählen / Anteil  ->  Grafik  ->  Aussage  ->  Aussage prüfen
(je Bewertung)     (Monat,          (mit Unsicher-                  („es wird    (Was müsste noch
                    Quelle ...)      heit)                           schlechter") stimmen?)
```

Der letzte Schritt ist der wichtigste, und der, der in der Praxis am häufigsten fehlt.

### Wie funktioniert Gruppieren?

Alles heute beruht auf einer einzigen Idee, die pandas `groupby` nennt:

```text
1. aufteilen       die Bewertungen nach einem Merkmal in Gruppen legen (z. B. Monat)
2. zusammenfassen  in jeder Gruppe dieselbe Rechnung machen (zählen, Anteil, Durchschnitt)
3. zusammensetzen  die Ergebnisse zu einer neuen, kleinen Tabelle verbinden
```

**Was passiert in der nächsten Zelle?** Wir nehmen nur sechs Bewertungen, damit man jeden Schritt sieht.

In [ ]:
mini = df[df["monat"].astype(str).isin(["2026-04", "2026-07"])].groupby("monat").head(3)[["monat", "sentiment"]]
mini = mini.sort_values("monat")                                 # je drei Bewertungen aus April und Juli
display(mini)

mini["ist_negativ"] = mini["sentiment"] == "negative"           # True = 1, False = 0
mini.groupby("monat")["ist_negativ"].agg(["sum", "count", "mean"])

**Wie lese ich das?** Oben sechs Bewertungen, drei aus dem April, drei aus dem Juli. Unten eine Zeile **pro Monat**:
- `sum` zählt die negativen (True zählt als 1),
- `count` zählt alle Bewertungen des Monats,
- `mean` ist `sum / count`, also der **Anteil** negativer Bewertungen.

Genau diese drei Zahlen berechnen wir gleich für alle 266 Bewertungen. Mehr Technik braucht es heute nicht: Wir ändern nur, **wonach** gruppiert wird (Monat, Quartal, Update-Phase, Quelle).

## 2. Wird es schlechter?

**Was passiert in der nächsten Zelle?** Wir zählen pro Monat, wie viele Bewertungen positiv, gemischt, neutral und negativ sind, als Tabelle und als gestapelte Balken.

In [ ]:
anzahl = pd.crosstab(df["monat"], df["sentiment"])[REIHENFOLGE]
display(anzahl.assign(gesamt=anzahl.sum(axis=1)))

ax = anzahl.plot(kind="bar", stacked=True, color=[FARBEN[s] for s in REIHENFOLGE], figsize=(9, 4), width=0.8)
ax.set_title("Bewertungen pro Monat (Anzahl)"); ax.set_xlabel(""); ax.set_ylabel("Anzahl Bewertungen")
ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Wie lese ich das?** Der rote Anteil wächst sichtbar. Aber schaut auf die Zahlen: Im Mai gab es **18** negative Bewertungen, im August **20**, fast gleich viele. War der August also kaum schlechter?

Nein. Im Mai kamen insgesamt 44 Bewertungen, im August nur 25. Die **Gesamtmenge schwankt** von Monat zu Monat, und damit auch jede Anzahl. Wer Monate vergleichen will, braucht den **Anteil**: 18 von 44 sind 41 %, 20 von 25 sind 80 %.

> **Nicht:** „Im August gab es 20 Beschwerden, im Mai 18."
> **Sondern:** „Im August war nur jede fünfte Bewertung nicht negativ, im Mai mehr als die Hälfte."

**Was passiert in der nächsten Zelle?** Dieselben Monate, aber jeder Balken auf 100 % gestreckt: Jetzt zeigt die Höhe den **Anteil** jeder Stimmung, nicht mehr die Anzahl.

In [ ]:
anteile = anzahl.div(anzahl.sum(axis=1), axis=0)
ax = anteile.plot(kind="bar", stacked=True, color=[FARBEN[s] for s in REIHENFOLGE], figsize=(9, 4), width=0.8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Bewertungen pro Monat (Anteil, jeder Monat = 100 %)"); ax.set_xlabel("")
ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Wie lese ich das?** Jetzt sieht man den Trend ohne Ablenkung durch die Menge: Rot wächst von einem schmalen Streifen im Februar auf vier Fünftel im August, Grün schrumpft entsprechend. Auch gemischte (gelb) und neutrale (grau) Bewertungen werden weniger.

Ein Nachteil dieser Darstellung: Man sieht nicht mehr, dass hinter dem Januar nur 5 und hinter dem April 53 Bewertungen stehen. Jeder Balken sieht gleich „wichtig" aus.

**Was passiert in der nächsten Zelle?** Wir konzentrieren uns auf eine Zahl, den Anteil negativer Bewertungen pro Monat, und zeigen dazu das Wilson-Intervall aus Sitzung 3: Wie sicher ist jeder Monatswert?

In [ ]:
neg = (df["sentiment"] == "negative").groupby(df["monat"]).agg(["sum", "count"])
neg.columns = ["negativ", "gesamt"]
neg["anteil"] = neg["negativ"] / neg["gesamt"]
neg[["von", "bis"]] = [radar.wilson(k, n) for k, n in zip(neg["negativ"], neg["gesamt"])]
display(neg.style.format({"anteil": "{:.0%}", "von": "{:.0%}", "bis": "{:.0%}"}))

def anteil_grafik(tabelle, titel):
    x = tabelle.index.astype(str)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(x, tabelle["von"], tabelle["bis"], color="#dc2626", alpha=0.15, label="95-%-Intervall (Wilson)")
    ax.plot(x, tabelle["anteil"], marker="o", color="#dc2626", label="Anteil negativ")
    for xi, n in zip(x, tabelle["gesamt"]):
        ax.annotate(f"n = {n}", (xi, 0.03), ha="center", fontsize=8, color="#6b7280")
    ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(titel); ax.legend(frameon=False, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False)
    return ax

anteil_grafik(neg, "Anteil negativer Bewertungen pro Monat")
plt.tight_layout(); plt.show()

**Wie lese ich das?** Der Anteil steigt von 10 % im Februar auf 80 % im August. Das ist ein deutlicher Trend, und die Intervalle von Februar und August überlappen nicht.

Aber schaut auf die Ränder:
- **Januar (n = 5)** und **September (n = 8):** Das Band ist riesig. Im September liegt der Anteil irgendwo zwischen etwa 20 % und 80 %.
- Der Knick im September auf 50 % sieht aus wie eine Erholung. Mit 8 Bewertungen ist das **nicht** zu unterscheiden von Zufall.

> **Nicht:** „Im September wird es endlich besser."
> **Sondern:** „Für September haben wir zu wenige Bewertungen, um etwas zu sagen."

Kleine Gruppen schwanken stark. Deshalb gehört zu jedem Anteil die Zahl dahinter (n) oder gleich das Intervall.

### Feiner: Wochen statt Monate

Wenn Monate schon unsicher sind, was passiert bei Wochen? Lena möchte schließlich schnell merken, wenn etwas kippt.

**Was passiert in der nächsten Zelle?** Wir gruppieren nach Kalenderwoche (`to_period("W")`) und zeichnen den Anteil negativer Bewertungen pro Woche.

In [ ]:
woche = df["datum"].dt.to_period("W")
wochen = (df["sentiment"] == "negative").groupby(woche).agg(["sum", "count"])
wochen.columns = ["negativ", "gesamt"]
wochen["anteil"] = wochen["negativ"] / wochen["gesamt"]

x = wochen.index.start_time
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(x, wochen["anteil"], marker=".", color="#dc2626", alpha=0.6, label="Anteil negativ pro Woche")
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"Anteil negativer Bewertungen pro Woche ({len(wochen)} Wochen, im Mittel {wochen['gesamt'].mean():.0f} Bewertungen)")
ax.legend(frameon=False, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Eine Zickzack-Linie. Mit im Mittel acht Bewertungen pro Woche springt der Anteil ständig zwischen 0 % und 100 %: Eine einzige Bewertung mehr oder weniger verschiebt ihn um mehr als 10 Prozentpunkte. Wer so eine Grafik jeden Montag anschaut, sieht jede Woche einen „Einbruch" oder eine „Erholung", die keine sind.

Ein üblicher Ausweg ist **Glätten**: Statt jede Woche einzeln nimmt man für jeden Zeitpunkt die letzten vier Wochen zusammen (*gleitender* oder *rollender* Durchschnitt).

**Was passiert in der nächsten Zelle?** Wir addieren für jede Woche die negativen und alle Bewertungen der letzten vier Wochen und teilen erst dann.

In [ ]:
vier = wochen[["negativ", "gesamt"]].rolling(4).sum()      # Summe über die letzten 4 Wochen
wochen["geglaettet"] = vier["negativ"] / vier["gesamt"]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(x, wochen["anteil"], marker=".", color="#dc2626", alpha=0.25, label="pro Woche")
ax.plot(x, wochen["geglaettet"], color="#dc2626", lw=2.5, label="geglättet (letzte 4 Wochen)")
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Anteil negativer Bewertungen: pro Woche und geglättet")
ax.legend(frameon=False, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Die dicke Linie zeigt den Trend, den man in der Zickzack-Linie nur ahnt: niedrig im Frühjahr, ab Mai steigend, im Sommer zwischen 55 und 80 %. Zwei Dinge kostet das Glätten:
- **Verzögerung:** Die geglättete Linie reagiert erst, wenn sich ein Wechsel über mehrere Wochen zeigt. Am rechten Rand sieht man es: Die Wochenwerte fallen im September, die geglättete Linie bleibt noch hoch. Umgekehrt würde ein echter Einbruch auch erst Wochen später sichtbar.
- **Wahl der Fensterbreite:** Vier Wochen sind eine Entscheidung, keine Wahrheit. Zwei Wochen wären unruhiger, acht noch träger.

Wichtig: Wir addieren erst die Anzahlen und teilen dann, statt die vier Wochen-Anteile zu mitteln. So zählt eine Woche mit 15 Bewertungen mehr als eine mit 2.

✏️ **Eure Aufgabe: Quartale statt Monate**

Monate mit 5 oder 8 Bewertungen sind zu klein. Fasst die Daten zu **Quartalen** zusammen und berechnet für jedes Quartal den Anteil negativer Bewertungen mit Intervall.

1. Füllt die Lücken `...` in der nächsten Zelle. Tipp: Für Monate haben wir `to_period("M")` benutzt.
2. Vergleicht mit der Monatsgrafik: Was wird besser, was geht verloren?

Die Lösung steht in der Zelle danach. Erst selbst versuchen.

In [ ]:
# Eure Lösung: Lücken ... füllen
try:                                   # try/except nur, damit „Alle ausführen" hier nicht stoppt
    df["quartal"] = df["datum"].dt.to_period(...)

    quartale = (df["sentiment"] == "negative").groupby(df[...]).agg(["sum", "count"])
    quartale.columns = ["negativ", "gesamt"]
    quartale["anteil"] = ...
    quartale[["von", "bis"]] = [radar.wilson(k, n) for k, n in zip(quartale["negativ"], quartale["gesamt"])]
    display(quartale)
except Exception as e:
    print("Noch nicht fertig, füllt die Lücken ... aus:", type(e).__name__)

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
df["quartal"] = df["datum"].dt.to_period("Q")

quartale = (df["sentiment"] == "negative").groupby(df["quartal"]).agg(["sum", "count"])
quartale.columns = ["negativ", "gesamt"]
quartale["anteil"] = quartale["negativ"] / quartale["gesamt"]
quartale[["von", "bis"]] = [radar.wilson(k, n) for k, n in zip(quartale["negativ"], quartale["gesamt"])]

anteil_grafik(quartale, "Anteil negativer Bewertungen pro Quartal")
plt.tight_layout(); plt.show()
quartale.style.format({"anteil": "{:.0%}", "von": "{:.0%}", "bis": "{:.0%}"})

**Wie lese ich das?** Drei Quartale, jedes mit 66 bis 129 Bewertungen: 20 % → 33 % → 68 %. Die Bänder sind viel schmaler, und der Trend ist eindeutig, weil keine Gruppe mehr winzig ist.

Was verloren geht: **wann** genau es kippt. In der Monatsgrafik sieht man, dass der Anstieg schon im Frühjahr beginnt; im Quartal steckt das im Mittelwert. Das ist immer der Tausch:

```text
feine Gruppen (Monat, Woche)    ->  mehr Detail, aber jeder Wert unsicher
grobe Gruppen (Quartal, Jahr)   ->  sicherere Werte, aber Details verschwinden
```

Es gibt keine richtige Größe. Wählt sie nach der Frage, und zeigt immer, wie viele Bewertungen in jeder Gruppe stecken.

### Wird es überall schlechter?

Eine Bewertung kann mehrere Themen nennen, jedes mit eigener Polarität (Sitzung 5). Statt ganzer Bewertungen zählen wir jetzt **Themen-Nennungen**: „Klang super, App stürzt ab" ergibt eine positive Nennung für Klang und eine negative für App & Bedienung. Gemischt gibt es auf dieser Ebene nicht mehr.

Achtung, die Einheit ändert sich: Aus 266 Bewertungen werden 409 Nennungen, eine Bewertung kann also mehrfach zählen. Und weil die Gruppen pro Thema klein werden, nehmen wir Quartale statt Monate.

**Was passiert in der nächsten Zelle?** Für jedes Thema und jedes Quartal: Welcher Anteil der Nennungen ist negativ? Die kleine Zahl im Feld ist die Anzahl der Nennungen.

In [ ]:
df["quartal"] = df["datum"].dt.to_period("Q")          # falls die Aufgabe übersprungen wurde
nennungen_q = pd.DataFrame([{"quartal": q, "thema": t, "negativ": p == "negativ"}
                            for q, liste in zip(df["quartal"], df["themen"]) for t, p in liste])
anteil_neg = nennungen_q.pivot_table(index="thema", columns="quartal", values="negativ", aggfunc="mean")
anzahl_n = nennungen_q.pivot_table(index="thema", columns="quartal", values="negativ", aggfunc="count")
anteil_neg = anteil_neg.sort_values(anteil_neg.columns[-1])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.imshow(anteil_neg.values, cmap="Reds", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(anteil_neg.shape[1])); ax.set_xticklabels(anteil_neg.columns.astype(str))
ax.set_yticks(range(len(anteil_neg))); ax.set_yticklabels(anteil_neg.index)
for i, thema in enumerate(anteil_neg.index):
    for j, q in enumerate(anteil_neg.columns):
        a, n = anteil_neg.loc[thema, q], anzahl_n.loc[thema, q]
        ax.text(j, i, f"{a:.0%}\nn = {n}", ha="center", va="center", fontsize=8,
                color="white" if a > 0.6 else "#111827")
ax.set_title("Anteil negativer Nennungen pro Thema und Quartal")
plt.tight_layout(); plt.show()
print(f"{len(nennungen_q)} Nennungen aus {len(df)} Bewertungen")

**Wie lese ich das?** Jede Zeile ein Thema, jede Spalte ein Quartal, je dunkler, desto mehr negative Nennungen. Fast überall wird es von links nach rechts dunkler: Akku, Geräuschunterdrückung, Verarbeitung und App kippen deutlich ins Negative.

Zwei Ausnahmen: **Klang** bleibt das beliebteste Thema (im dritten Quartal immer noch mehr Lob als Kritik), und **Tragekomfort** verändert sich kaum. Bei Tragekomfort stehen im dritten Quartal aber nur 9 Nennungen dahinter. Ob das wirklich stabil ist, können wir nicht sagen.

Das passt zu Abschnitt 4: Der Anstieg kommt nicht von **einem** Thema, sondern betrifft fast alle.

## 3. Und die Sterne?

Lena fragt: *„Wozu die ganze Textanalyse? Sterne haben wir doch schon."* Eine berechtigte Frage, denn Sterne kosten nichts.

**Was passiert in der nächsten Zelle?** Zuerst: Passen die Sterne überhaupt zum Text? Eine Kreuztabelle zählt für jede Sternezahl, wie viele Bewertungen welche Stimmung haben. „ohne" heißt: keine Sterne angegeben.

In [ ]:
sterne_text = pd.crosstab(df["sterne"].astype("float").fillna(0).map(lambda s: "ohne" if s == 0 else f"{s:.0f} ★"),
                          df["sentiment"])[REIHENFOLGE]
sterne_text = sterne_text.loc[["1 ★", "2 ★", "3 ★", "4 ★", "5 ★", "ohne"]]
sterne_text.style.background_gradient(cmap="Blues", axis=None)

**Wie lese ich das?** Sterne und Text passen gut zusammen: 4 und 5 Sterne sind nie negativ, 1 Stern ist immer negativ. Gemischte Bewertungen verteilen sich auf 2 bis 4 Sterne, man kann sie an den Sternen also nicht erkennen.

Und die letzte Zeile: **33 negative Bewertungen haben gar keine Sterne**, fast jede dritte der 102. Bei den positiven sind es nur 6 von 106. Das wird gleich wichtig.

**Was passiert in der nächsten Zelle?** Links der Sterne-Durchschnitt pro Monat, rechts der Anteil der Bewertungen **ohne** Sterne. Die gestrichelte Linie zeigt den echten Durchschnitt aus der Wahrheit, also mit den fehlenden Sternen. Den gibt es nur im Kurs.

In [ ]:
sterne_wahr = wahrheit.loc[df["review_id"], "sterne_wahr"].values
sterne = pd.DataFrame({"monat": df["monat"], "sterne": df["sterne"].astype("float"), "wahr": sterne_wahr})
sterne = sterne.groupby("monat").agg(mittel=("sterne", "mean"), wahr=("wahr", "mean"),
                                     fehlt=("sterne", lambda s: s.isna().mean()))
x = sterne.index.astype(str)

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 3.8))
links.plot(x, sterne["mittel"], marker="o", color="#2563eb", label="Ø Sterne (was wir sehen)")
links.plot(x, sterne["wahr"], ls="--", color="#2563eb", alpha=0.5, label="Ø Sterne (wahr, nur im Kurs)")
links.set_ylim(1, 5); links.set_title("Durchschnittliche Sterne pro Monat"); links.legend(frameon=False, fontsize=8)
rechts.bar(x, sterne["fehlt"], color="#9ca3af")
rechts.set_ylim(0, 0.5); rechts.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
rechts.set_title("Anteil Bewertungen ohne Sterne")
for ax in (links, rechts):
    ax.tick_params(axis="x", rotation=45); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
sterne.round(2)

**Wie lese ich das?**
- **Die Sterne zeigen denselben Trend:** von 3,9 im Februar auf 1,8 im August. Für die Frage *„Wird es schlechter?"* reicht das billige Signal also. Das ist eine gute Nachricht, und ein Grund, Sterne immer mitzuzeigen.
- **Aber der Anteil ohne Sterne steigt** von etwa 7 % auf etwa 25 %. Aus Sitzung 4 wissen wir: Negative Bewertungen haben deutlich häufiger keine Sterne. Der Durchschnitt wird also aus dem freundlicheren Rest berechnet und liegt **zu hoch**. Die gestrichelte Linie zeigt es: Ab März liegt der sichtbare Durchschnitt über dem wahren, im Juli etwa 2,8 statt 2,5. Je mehr negative Bewertungen, desto größer der Fehler.
- **Und das Wichtigste:** Zwei Sterne sagen *dass* jemand unzufrieden ist, nicht *womit*. Für Lenas eigentliche Frage, *warum*, brauchen wir den Text.

> **Nicht:** „Sterne reichen, die Textanalyse sparen wir uns."
> **Sondern:** Sterne für den schnellen Trend, der Text für das Warum, und fehlende Werte immer mitzeigen.

## 4. War es das Update?

Lena ist sich sicher: *„Am 1. Juli kam App-Update 2.0. Seitdem geht es bergab. Das Update war schuld."*

**Was passiert in der nächsten Zelle?** Wir teilen die Bewertungen in *vor* und *nach* dem 1. Juli und vergleichen den Anteil negativer Bewertungen, mit Intervall.

In [ ]:
UPDATE = pd.Timestamp("2026-07-01")
df["phase"] = np.where(df["datum"] < UPDATE, "vor dem Update", "nach dem Update")

phasen = (df["sentiment"] == "negative").groupby(df["phase"]).agg(["sum", "count"])
phasen = phasen.loc[["vor dem Update", "nach dem Update"]]
for phase, (k, n) in phasen.iterrows():
    von, bis = radar.wilson(k, n)
    print(f"{phase:16}  {k / n:4.0%} negativ   (Intervall {von:.0%} bis {bis:.0%}, n = {n})")

**Wie lese ich das?** Vor dem Update 28 % negativ, danach 68 %. Die Intervalle überlappen nicht, der Unterschied ist also kein Zufall. Ist das der Beweis?

Noch nicht. Bisher wissen wir nur: **Nach** dem Update ist es schlechter. Ob es schlechter ist, **weil** das Update kam, ist eine andere Behauptung. Wenn das Update die Ursache wäre, müssten zwei weitere Dinge stimmen:

```text
Behauptung: Das Update hat die Bewertungen verschlechtert.
   Folgerung 1: Vor dem 1. Juli war alles stabil, der Anstieg beginnt erst danach.
   Folgerung 2: Die zusätzlichen Beschwerden betreffen vor allem die App.
```

Prüfen wir beide.

**Was passiert in der nächsten Zelle?** Folgerung 1: die Monatsgrafik von vorhin, mit einer Linie am Update-Datum.

In [ ]:
ax = anteil_grafik(neg, "Anteil negativer Bewertungen pro Monat, mit Update-Datum")
ax.axvline(5.5, color="#111827", ls="--")                 # zwischen Juni und Juli
ax.annotate("App-Update 2.0", (5.6, 0.92), fontsize=9)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Der Anstieg beginnt **lange vor** dem Update: Von 10 % im Februar geht es über 41 % im Mai auf 50 % im Juni. Das Update fällt mitten in einen Trend, der schon läuft. Ein Vorher-Nachher-Vergleich findet bei jedem laufenden Trend einen Unterschied, egal welches Datum man wählt.

**Folgerung 1 stimmt nicht.**

**Was passiert in der nächsten Zelle?** Eine Gegenprobe: Wir tun so, als wäre das Update an einem anderen Tag gekommen, und machen denselben Vorher-Nachher-Vergleich für sechs verschiedene Daten.

In [ ]:
probe_daten = ["2026-03-01", "2026-04-01", "2026-05-01", "2026-06-01", "2026-07-01", "2026-08-01"]
zeilen = []
for d in probe_daten:
    vor = df["datum"] < pd.Timestamp(d)
    zeilen.append({"angebliches Update": pd.Timestamp(d).strftime("%d.%m."),
                   "vorher negativ": (df.loc[vor, "sentiment"] == "negative").mean(),
                   "nachher negativ": (df.loc[~vor, "sentiment"] == "negative").mean()})
gegenprobe = pd.DataFrame(zeilen).set_index("angebliches Update")

ax = gegenprobe.plot(kind="bar", color=["#9ca3af", "#dc2626"], figsize=(9, 3.8), width=0.8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylim(0, 1)
ax.set_title("Vorher-Nachher-Vergleich für sechs angebliche Update-Daten"); ax.set_xlabel("")
ax.legend(frameon=False, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
gegenprobe.style.format("{:.0%}")

**Wie lese ich das?** Egal welches Datum wir nehmen: Nachher ist es **immer** deutlich schlechter als vorher, beim 1. März genauso wie beim 1. August. Hätte das Update am 1. April stattgefunden, sähe der Vergleich genauso „überzeugend" aus.

Das liegt nicht am Update, sondern am Trend: Wenn es ohnehin jeden Monat schlechter wird, ist „nachher" immer schlechter als „vorher". Ein Vorher-Nachher-Vergleich kann ein Ereignis nur dann belegen, wenn es **vorher stabil** war.

**Was passiert in der nächsten Zelle?** Folgerung 2: Welcher Anteil der Beschwerden betrifft welches Thema, vor und nach dem Update? Dafür zerlegen wir jede Bewertung in ihre Themen-Nennungen (eine Bewertung kann mehrere haben) und behalten die negativen.

In [ ]:
nennungen = pd.DataFrame([{"phase": ph, "thema": t, "polaritaet": p}
                          for ph, liste in zip(df["phase"], df["themen"]) for t, p in liste])
beschwerden = nennungen[nennungen["polaritaet"] == "negativ"]

zaehl = pd.crosstab(beschwerden["thema"], beschwerden["phase"])[["vor dem Update", "nach dem Update"]]
anteil_thema = zaehl / zaehl.sum()                       # jede Spalte = 100 % der Beschwerden
anteil_thema = anteil_thema.sort_values("nach dem Update")

ax = anteil_thema.plot(kind="barh", color=["#9ca3af", "#dc2626"], figsize=(8, 4.5), width=0.8)
ax.xaxis.set_major_locator(plt.MultipleLocator(0.05))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Welcher Anteil der Beschwerden betrifft welches Thema?"); ax.set_ylabel("")
ax.legend(frameon=False, loc="lower right"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

app = zaehl.loc["App & Bedienung"]
for phase in zaehl.columns:
    k, n = app[phase], zaehl[phase].sum()
    von, bis = radar.wilson(k, n)
    print(f"App & Bedienung {phase:16}: {k:2} von {n:3} Beschwerden = {k / n:.0%}  (Intervall {von:.0%} bis {bis:.0%})")

**Wie lese ich das?** Wenn das Update schuld wäre, müsste der rote Balken bei **App & Bedienung** deutlich länger sein als der graue. Er ist etwas länger: 18 % → 21 %. Und App & Bedienung ist nach dem Update das häufigste Beschwerdethema. Aber:
- Das war es **vorher auch** schon.
- Die Intervalle überlappen fast vollständig (etwa 12–25 % gegen 13–31 %). Bei 16 Nennungen nach dem Update ist das im Rauschen.
- Andere Themen bewegen sich genauso stark: **Akku & Laden** steigt von 15 % auf 20 %, **Tragekomfort** fällt von 12 % auf 4 %. Hat das App-Update den Tragekomfort verbessert? Sicher nicht. Bei kleinen Zahlen wackeln alle Balken.

**Folgerung 2 stimmt auch nicht.** Der Anstieg verteilt sich auf alle Themen, wie schon in Sitzung 5 beim Vergleich Q1 gegen Q3.

**Ergebnis:** Nach dem Update ist es schlechter, aber das Update **erklärt** den Anstieg nicht. Was ihn erklärt, sagen diese Daten nicht. Mögliche Ursachen, die genauso zu den Zahlen passen:
- eine neue Käufergruppe mit höheren Erwartungen (z. B. durch eine Werbeaktion im Frühjahr),
- eine Produktionscharge mit mehr Defekten,
- ein Konkurrenzprodukt, gegen das die Nimbus Q2 jetzt schlechter aussieht.

Was helfen würde: die **App-Version** in jeder Bewertung, eine **Vergleichsgruppe** ohne Update (z. B. gestaffelter Rollout), oder Bewertungen, die das Update ausdrücklich nennen.

Und die Auflösung: In unserem Generator gibt es gar kein Update. Der Anstieg ist eingebaut, negative Bewertungen kommen dort einfach später im Jahr. Genau deshalb können wir hier prüfen, ob unsere Methode uns in die Irre führt: Der Vorher-Nachher-Vergleich allein hätte es getan.

> 💡 **Kernpunkt:** Aggregierte Zahlen zeigen **Muster**, keine **Ursachen**.
>
> **Nicht:** „Nach dem Update schlechter, also wegen des Updates."
> **Sondern:** Erst fragen, was **sonst noch** stimmen müsste, wenn die Ursache stimmt (Wann beginnt es? Wo zeigt es sich?), und genau das prüfen.

## 5. Eure Manager-Frage

Lena hat noch eine Frage: *„Unsere Bewertungen kommen aus drei Quellen: Webshop (`web`), deutsche App (`app_de`) und US-App (`app_us`). Sind die US-Nutzer unzufriedener? Sollen wir dort zuerst etwas tun?"*

✏️ **Eure Aufgabe:**
1. Füllt die Lücken `...` und berechnet für jede Quelle den Anteil negativer Bewertungen mit Intervall. Tipp: Es ist dieselbe Rechnung wie für Monate, nur nach einer anderen Spalte gruppiert.
2. Schreibt **einen Satz** als Antwort an Lena.
3. Schreibt **einen Satz**, was die Zahl **nicht** beweist.

In [ ]:
# Eure Lösung: Lücken ... füllen
try:                                   # try/except nur, damit „Alle ausführen" hier nicht stoppt
    pro_quelle = (df["sentiment"] == "negative").groupby(df[...]).agg(["sum", "count"])
    pro_quelle.columns = ["negativ", "gesamt"]
    pro_quelle["anteil"] = ...
    pro_quelle[["von", "bis"]] = [radar.wilson(k, n) for k, n in zip(pro_quelle["negativ"], pro_quelle["gesamt"])]
    display(pro_quelle)
except Exception as e:
    print("Noch nicht fertig, füllt die Lücken ... aus:", type(e).__name__)

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
pro_quelle = (df["sentiment"] == "negative").groupby(df["quelle"]).agg(["sum", "count"])
pro_quelle.columns = ["negativ", "gesamt"]
pro_quelle["anteil"] = pro_quelle["negativ"] / pro_quelle["gesamt"]
pro_quelle[["von", "bis"]] = [radar.wilson(k, n) for k, n in zip(pro_quelle["negativ"], pro_quelle["gesamt"])]
pro_quelle.style.format({"anteil": "{:.0%}", "von": "{:.0%}", "bis": "{:.0%}"})

**Was passiert in der nächsten Zelle?** Dasselbe als Grafik: ein Balken pro Quelle, die Striche zeigen das Intervall.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(pro_quelle.index, pro_quelle["anteil"], color="#dc2626", alpha=0.8)
ax.errorbar(pro_quelle["anteil"], pro_quelle.index,
            xerr=[pro_quelle["anteil"] - pro_quelle["von"], pro_quelle["bis"] - pro_quelle["anteil"]],
            fmt="none", ecolor="#111827", capsize=4)
for i, (a, n) in enumerate(zip(pro_quelle["anteil"], pro_quelle["gesamt"])):
    ax.annotate(f"{a:.0%}  (n = {n})", (0.02, i), va="center", color="white", fontsize=9)
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Anteil negativer Bewertungen pro Quelle"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** In der US-App sind 53 % der Bewertungen negativ, in der deutschen App 31 %, im Webshop 38 %. Die Intervalle von `app_us` und `app_de` überlappen kaum. Eine naheliegende Antwort an Lena: *„Ja, die US-Nutzer sind deutlich unzufriedener."*

**Die Auflösung:** In unserem Generator wird die Quelle **zufällig** vergeben, unabhängig von der Stimmung. Es gibt keinen echten Unterschied zwischen den Quellen. Was ihr seht, ist Zufall, und er sieht überzeugend aus.

Glaubt ihr nicht? Dann mischen wir selbst.

**Was passiert in der nächsten Zelle?** Wir verteilen die Quellen-Etiketten **zufällig neu** auf die Bewertungen: Jede Quelle behält ihre Größe, aber welche Bewertung welche Quelle bekommt, entscheidet der Zufall. Jeder echte Zusammenhang zwischen Quelle und Stimmung ist damit sicher zerstört. Führt die Zelle mehrmals aus.

In [ ]:
gemischt = np.random.permutation(df["quelle"].values)           # Quellen zufällig neu verteilen
ergebnis = (df["sentiment"] == "negative").groupby(gemischt).mean().sort_values()
for quelle, anteil in ergebnis.items():
    print(f"{quelle:7} {anteil:4.0%} negativ  " + "█" * int(anteil * 50))
print(f"\nAbstand zwischen bester und schlechtester Quelle: {ergebnis.max() - ergebnis.min():.0%}")

**Wie lese ich das?** Bei jedem Lauf sieht es anders aus, und oft liegt eine andere Quelle vorn, obwohl es nach dem Mischen garantiert keinen Zusammenhang mehr gibt. Abstände von 10 Prozentpunkten und mehr entstehen ganz von selbst, einfach weil die Gruppen klein sind.

**Was passiert in der nächsten Zelle?** Wir mischen nicht einmal, sondern **1000-mal**, und schauen, wie oft der Abstand so groß wird wie in den echten Daten (23 Prozentpunkte).

In [ ]:
echt = pro_quelle["anteil"].max() - pro_quelle["anteil"].min()
ist_negativ = df["sentiment"] == "negative"
rng = np.random.default_rng(0)                                 # fester Startwert: alle sehen dieselbe Grafik
abstaende = []
for _ in range(1000):
    a = ist_negativ.groupby(rng.permutation(df["quelle"].values)).mean()
    abstaende.append(a.max() - a.min())
abstaende = np.array(abstaende)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(abstaende, bins=30, color="#9ca3af")
ax.axvline(echt, color="#dc2626", lw=2)
ax.annotate(f"echte Daten: {echt:.0%}", (echt + 0.005, ax.get_ylim()[1] * 0.85), color="#dc2626")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Abstand zwischen bester und schlechtester Quelle bei 1000 zufälligen Mischungen")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
print(f"In {(abstaende >= echt).mean():.1%} der Mischungen war der Abstand mindestens so groß wie in den echten Daten.")

**Wie lese ich das?** Die grauen Balken zeigen, welche Abstände reiner Zufall erzeugt: meistens 5 bis 15 Prozentpunkte. Die rote Linie liegt weit rechts, nur in etwa 3 von 100 Mischungen entsteht ein so großer Abstand. Unsere Quellen hatten also **Pech**: ein seltener, aber kein unmöglicher Zufall.

Und jetzt zählt mal, wie viele Vergleiche wir heute schon gemacht haben: Monate, Wochen, Quartale, Update-Phasen, acht Themen, drei Quellen. Bei so vielen Vergleichen ist es fast sicher, dass **einer** davon so ein seltener Zufall ist. Das heißt nicht, dass jede Auffälligkeit Zufall ist. Aber ein einzelnes auffälliges Segment ist ein Hinweis, kein Befund.

Was man in echten Daten tun würde, wo niemand in den Generator schauen kann:
- **Mit neuen Daten prüfen:** Ist `app_us` auch nächsten Monat auffällig?
- **Nach einem Mechanismus fragen:** Gibt es einen Grund? Andere App-Version, anderer Versand, andere Erwartungen?
- **Nicht sofort handeln:** erst bestätigen, dann Budget verschieben.

> **Nicht:** „Ein Segment fällt auf, also handeln wir dort."
> **Sondern:** „Ein Segment fällt auf: Wir prüfen mit neuen Daten, ob es bleibt."

## 6. Der Bericht an Lena

Lena hat keine Zeit für 15 Grafiken. Sie braucht eine Seite: die Antworten auf ihre drei Fragen, jeweils mit der wichtigsten Zahl, und was wir **nicht** wissen.

**Was passiert in der nächsten Zelle?** Wir setzen die Ergebnisse von heute zu einem kurzen Bericht zusammen. Alle **Zahlen** kommen aus den Variablen oben, keine ist von Hand eingetippt; ändert sich der Datensatz, ändern sich die Zahlen mit. Die **Sätze** dazu haben wir selbst geschrieben, nach den Prüfungen von heute. Die müsste man bei neuen Daten neu prüfen.

In [ ]:
q = quartale["anteil"]
vor_k, vor_n = phasen.loc["vor dem Update"]
nach_k, nach_n = phasen.loc["nach dem Update"]
sterne_q = df["sterne"].astype("float").groupby(df["quartal"]).agg(mittel="mean", fehlt=lambda s: s.isna().mean())

bericht = f"""
## Bericht: Nimbus Q2, Bewertungen Januar bis September 2026
Grundlage: {len(df)} Bewertungen mit Datum ({ohne_datum} ohne Datum nicht berücksichtigt)

**1. Wird es schlechter?** Ja, deutlich. Negativer Anteil pro Quartal: {" → ".join(f"{a:.0%}" for a in q)}.
Einzelne Monate am Rand (Januar, September) haben zu wenige Bewertungen für eine Aussage.

**2. Reichen die Sterne?** Für den Trend ja: Ø {" → ".join(f"{m:.1f}".replace(".", ",") for m in sterne_q["mittel"])} Sterne pro Quartal.
Aber der Anteil ohne Sterne steigt ({" → ".join(f"{f:.0%}" for f in sterne_q["fehlt"])}), der Durchschnitt wirkt deshalb zu freundlich.
Und Sterne sagen nicht, *woran* es liegt.

**3. War es das App-Update?** Nach dem Update {nach_k / nach_n:.0%} negativ, vorher {vor_k / vor_n:.0%}. Das Update erklärt das aber **nicht**:
der Anstieg beginnt schon im Frühjahr, und Beschwerden über die App nehmen nicht stärker zu als andere.
Die Ursache ist mit diesen Daten nicht zu klären. Hilfreich wären: App-Version je Bewertung, eine Vergleichsgruppe ohne Update.

**Nicht belegt:** Unterschiede zwischen den Quellen (Web, App DE, App US). Die US-App fällt auf, das ist aber mit Zufall vereinbar; erst mit neuen Daten prüfen.
"""
display(Markdown(bericht))

**Wie lese ich das?** Drei Fragen, drei Antworten, und eine Zeile dazu, was wir **nicht** wissen. Das ist der Unterschied zwischen einer Grafik und einer Analyse: Die Grafik zeigt ein Muster, der Bericht sagt, was es bedeutet und wie sicher es ist.

Nächste Woche bauen wir daraus ein Dashboard, und lassen den Text vom LLM schreiben. Dann gilt alles aus Sitzung 6: Wir prüfen, ob die Zusammenfassung zu den Zahlen passt.

## Zusammenfassung

```text
1. „Mehr Beschwerden im Monat X"       -> Anteil statt Anzahl: die Gesamtmenge schwankt
2. Monat mit wenigen Bewertungen       -> n und Intervall mitzeigen; kleine Gruppen schwanken stark
3. Woche, Monat oder Quartal?          -> fein = Detail, aber unsicher; grob = sicher, aber ungenau;
                                          Glätten zeigt den Trend, reagiert aber verzögert
4. Sterne als billiges Signal          -> zeigen den Trend, nicht das Warum; fehlende Sterne verzerren
5. „Nach X schlechter, also wegen X"   -> Folgerungen prüfen: Wann beginnt es? Wo zeigt es sich?
                                          Bei laufendem Trend ist „nachher" immer schlechter
6. Ein Segment fällt auf               -> kann Zufall sein (Mischen zeigt, wie groß Zufall wird);
                                          mit neuen Daten bestätigen, bevor man handelt
7. Ergebnis für die Managerin          -> eine Seite: Antwort, wichtigste Zahl, was nicht belegt ist
```

## Zum Weiterprobieren (freiwillig)

1. **Fensterbreite:** Ändert beim Glätten `rolling(4)` auf `rolling(2)` und `rolling(8)`. Wie verändern sich Ruhe und Verzögerung?
2. **Ein Thema über die Zeit:** Wie entwickelt sich der Anteil der Beschwerden über **Akku & Laden** pro Quartal? Ist der Anstieg nach dem Update hier echt oder Zufall? (Tipp: Intervall berechnen.)
3. **Mischen für Monate:** Verteilt die Monate zufällig neu, wie in Abschnitt 5 die Quellen. Wie sieht der Monatsverlauf dann aus? Was sagt das über unseren echten Trend?
4. **Sterne-Lücke schließen:** Füllt die fehlenden Sterne wie in Sitzung 4 nach Stimmungsgruppe und vergleicht den Durchschnitt mit der gestrichelten Linie.

## Ausblick

**Nächste Woche (25.11):** der **Meilenstein**. Wir setzen alles zusammen, von den Rohdaten über die echte Pipeline (Stimmung, Themen, verlässliches JSON) bis zu einem Insight-Dashboard mit einer Zusammenfassung durch das LLM.